## 🎯 Learning Objectives
* Understand the architecture and components of a multi-document query engine in LlamaIndex.
* Implement a multi-document query engine using LlamaIndex's RouterQueryEngine and QueryEngineTool.
* Develop effective routing strategies to direct queries to the most relevant document indices.
* Evaluate the routing and retrieval performance of a multi-document RAG system.


## Exercise: Build a Multi-Document Query Engine

### Task Description

In real-world RAG applications, information often resides across numerous distinct documents or knowledge bases. A single vector store might not be optimal for all scenarios, especially when queries require specific expertise or context from a particular document. This exercise challenges you to build a multi-document query engine using LlamaIndex that can intelligently route incoming queries to the most appropriate underlying document index.

Your goal is to simulate a scenario where you have information spread across several distinct documents. Your query engine should be able to determine which document(s) are most relevant to a user's question and retrieve answers exclusively from those selected sources.

### Requirements

1.  **Mock Dataset Creation**: Create at least three distinct mock documents (e.g., text files) with clearly separable content. Each document should cover a different topic or aspect.
2.  **Individual Document Indices**: For each mock document, create a separate `VectorStoreIndex` instance. These indices will serve as the individual knowledge bases.
3.  **Query Engine Tools**: Wrap each `VectorStoreIndex` with a `QueryEngineTool`. Provide a clear and descriptive `metadata` for each tool, which the router will use to decide routing.
4.  **Router Query Engine**: Implement a `RouterQueryEngine` that uses a `SelectorComponent` (e.g., `LLMSingleSelector` or `PydanticMultiSelector`) to choose the most relevant `QueryEngineTool` based on the input query.
5.  **Query and Evaluation**: Pose several diverse questions to your `RouterQueryEngine`. Some questions should clearly target a single document, while others might be ambiguous or require information from multiple (if you choose a multi-selector). Print the response and, crucially, identify which underlying document index was used for retrieval.
6.  **Robustness**: Ensure your routing logic is robust enough to handle queries that might not perfectly match any single document's description, demonstrating a reasonable fallback or best-guess mechanism.

### Evaluation Criteria

*   **Correctness**: Does the `RouterQueryEngine` consistently route queries to the correct document index?
*   **Clarity of Routing Logic**: Are the `QueryEngineTool` descriptions clear and effective for the selector?
*   **Code Quality**: Is the code well-structured, readable, and appropriately commented?
*   **Demonstration**: Are the test queries diverse and do they effectively showcase the multi-document routing capability?
*   **LlamaIndex Best Practices**: Does the solution leverage LlamaIndex components effectively and follow recommended patterns for multi-document RAG?


In [ ]:
import os
import logging
from pathlib import Path

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, ServiceContext, Settings
from llama_index.core.llms import LLM
from llama_index.llms.openai import OpenAI
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.tools import QueryEngineTool
from llama_index.embeddings.openai import OpenAIEmbedding

# Set up logging for better visibility into LlamaIndex operations
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- Configuration and Setup ---

# Ensure you have your OpenAI API key set as an environment variable
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Initialize LLM and Embedding Model (using modern LlamaIndex Settings)
# In 2026, we assume advanced, cost-effective models are standard.
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.1)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Create a temporary directory for mock documents
data_dir = Path("./mock_docs")
data_dir.mkdir(exist_ok=True)

# --- Create Mock Documents ---

# Document 1: Information about renewable energy
doc1_content = """
Renewable energy sources are crucial for combating climate change. Solar power, generated from sunlight, is becoming increasingly efficient with advancements in photovoltaic cell technology. Wind energy harnesses kinetic energy from wind through turbines. Hydropower, derived from flowing water, is a reliable source but often requires large-scale infrastructure. Geothermal energy utilizes heat from the Earth's core. These sources offer sustainable alternatives to fossil fuels.
"""
(data_dir / "renewable_energy.txt").write_text(doc1_content)

# Document 2: Information about quantum computing
doc2_content = """
Quantum computing leverages principles of quantum mechanics, such as superposition and entanglement, to perform computations. Unlike classical bits that represent 0 or 1, quantum bits (qubits) can exist in multiple states simultaneously. This allows quantum computers to solve certain complex problems exponentially faster than classical computers. Major challenges include qubit stability, error correction, and scalability. Companies like IBM and Google are at the forefront of quantum hardware development.
"""
(data_dir / "quantum_computing.txt").write_text(doc2_content)

# Document 3: Information about ancient Roman history
doc3_content = """
The Roman Empire was a vast and powerful civilization that dominated much of Europe, North Africa, and the Middle East for over a thousand years. Founded in 27 BC with Augustus as its first emperor, it was known for its advanced engineering, legal systems, and military prowess. Key figures include Julius Caesar, Nero, and Constantine the Great. The empire eventually split into Western and Eastern halves, with the Western Roman Empire falling in 476 AD, marking the end of ancient history in Europe.
"""
(data_dir / "roman_history.txt").write_text(doc3_content)

print(f"Mock documents created in: {data_dir}")

# --- Helper Function to Load and Index a Single Document ---
def create_index_from_file(file_path: Path, index_name: str) -> VectorStoreIndex:
    """Loads a single document and creates a VectorStoreIndex from it."""
    print(f"Loading document: {file_path.name}")
    documents = SimpleDirectoryReader(input_files=[file_path]).load_data()
    index = VectorStoreIndex.from_documents(documents)
    print(f"Created index '{index_name}' from {file_path.name}")
    return index

print("Setup complete. Ready for your implementation.")


## Your Turn! Implement the Multi-Document Query Engine

Now it's your turn to build the `RouterQueryEngine`.

Using the `create_index_from_file` helper function and the mock documents provided in the setup cell, follow these steps:

1.  **Create individual `VectorStoreIndex` instances** for each of the mock documents (`renewable_energy.txt`, `quantum_computing.txt`, `roman_history.txt`).
2.  **Create `QueryEngineTool` instances** for each index. Remember to provide a descriptive `metadata` for each tool that clearly explains what kind of questions it can answer. This description is crucial for the `LLMSingleSelector` to make informed routing decisions.
3.  **Instantiate `RouterQueryEngine`**: Use `LLMSingleSelector` as your selector component and pass your list of `QueryEngineTool` instances to the router.
4.  **Test your engine**: Ask at least 5-7 diverse questions. For each question, print the query, the response, and try to infer or explicitly state which document index was used (you can often see this in the source nodes or by observing the response content).

Good luck!


In [ ]:
import os
import logging
from pathlib import Path

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, ServiceContext, Settings
from llama_index.llms.openai import OpenAI
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.tools import QueryEngineTool
from llama_index.embeddings.openai import OpenAIEmbedding

# Re-run setup for completeness if cell is run independently
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Ensure you have your OpenAI API key set as an environment variable
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Initialize LLM and Embedding Model (using modern LlamaIndex Settings)
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.1)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Create a temporary directory for mock documents
data_dir = Path("./mock_docs")
data_dir.mkdir(exist_ok=True)

# --- Create Mock Documents (if not already created by setup cell) ---
# This block ensures the files exist even if the setup cell wasn't run first
if not (data_dir / "renewable_energy.txt").exists():
    doc1_content = """
    Renewable energy sources are crucial for combating climate change. Solar power, generated from sunlight, is becoming increasingly efficient with advancements in photovoltaic cell technology. Wind energy harnesses kinetic energy from wind through turbines. Hydropower, derived from flowing water, is a reliable source but often requires large-scale infrastructure. Geothermal energy utilizes heat from the Earth's core. These sources offer sustainable alternatives to fossil fuels.
    """
    (data_dir / "renewable_energy.txt").write_text(doc1_content)

if not (data_dir / "quantum_computing.txt").exists():
    doc2_content = """
    Quantum computing leverages principles of quantum mechanics, such as superposition and entanglement, to perform computations. Unlike classical bits that represent 0 or 1, quantum bits (qubits) can exist in multiple states simultaneously. This allows quantum computers to solve certain complex problems exponentially faster than classical computers. Major challenges include qubit stability, error correction, and scalability. Companies like IBM and Google are at the forefront of quantum hardware development.
    """
    (data_dir / "quantum_computing.txt").write_text(doc2_content)

if not (data_dir / "roman_history.txt").exists():
    doc3_content = """
    The Roman Empire was a vast and powerful civilization that dominated much of Europe, North Africa, and the Middle East for over a thousand years. Founded in 27 BC with Augustus as its first emperor, it was known for its advanced engineering, legal systems, and military prowess. Key figures include Julius Caesar, Nero, and Constantine the Great. The empire eventually split into Western and Eastern halves, with the Western Roman Empire falling in 476 AD, marking the end of ancient history in Europe.
    """
    (data_dir / "roman_history.txt").write_text(doc3_content)

# --- Helper Function to Load and Index a Single Document ---
def create_index_from_file(file_path: Path, index_name: str) -> VectorStoreIndex:
    """Loads a single document and creates a VectorStoreIndex from it."""
    print(f"Loading document: {file_path.name}")
    documents = SimpleDirectoryReader(input_files=[file_path]).load_data()
    index = VectorStoreIndex.from_documents(documents)
    print(f"Created index '{index_name}' from {file_path.name}")
    return index


# --- Reference Solution Implementation ---

print("\n--- Building Multi-Document Query Engine (Reference Solution) ---")

# 1. Create individual VectorStoreIndex instances
energy_index = create_index_from_file(data_dir / "renewable_energy.txt", "Renewable Energy Index")
quantum_index = create_index_from_file(data_dir / "quantum_computing.txt", "Quantum Computing Index")
roman_index = create_index_from_file(data_dir / "roman_history.txt", "Roman History Index")

# 2. Create QueryEngineTool instances
# The 'description' is critical for the LLMSingleSelector to make routing decisions.
energy_tool = QueryEngineTool(
    query_engine=energy_index.as_query_engine(),
    metadata={
        "name": "renewable_energy_tool",
        "description": "Provides information about various renewable energy sources like solar, wind, hydro, and geothermal power. Use this for questions about sustainable energy, climate change solutions, or energy technologies."
    }
)

quantum_tool = QueryEngineTool(
    query_engine=quantum_index.as_query_engine(),
    metadata={
        "name": "quantum_computing_tool",
        "description": "Answers questions related to quantum mechanics, quantum computing principles (qubits, superposition, entanglement), quantum hardware, and challenges in the field. Use this for advanced computing or physics topics."
    }
)

roman_tool = QueryEngineTool(
    query_engine=roman_index.as_query_engine(),
    metadata={
        "name": "roman_history_tool",
        "description": "Contains historical facts about the Roman Empire, its emperors, key events, engineering, and its eventual decline. Use this for questions about ancient history, Roman civilization, or historical figures like Julius Caesar."
    }
)

# Collect all tools into a list
query_engine_tools = [
    energy_tool,
    quantum_tool,
    roman_tool
]

# 3. Instantiate RouterQueryEngine
# LLMSingleSelector uses the LLM to pick the best tool based on the query and tool descriptions.
router_query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=query_engine_tools,
    verbose=True # Set to True to see the routing decisions made by the LLM
)

print("\nRouterQueryEngine initialized with 3 tools.")

# 4. Test your engine with diverse questions
print("\n--- Testing RouterQueryEngine ---")

queries = [
    "What are the main types of renewable energy sources?",
    "Explain the concept of superposition in quantum computing.",
    "Who was the first emperor of the Roman Empire?",
    "How do wind turbines generate electricity?",
    "What are the primary challenges in building scalable quantum computers?",
    "Tell me about Julius Caesar.",
    "What is geothermal energy?",
    "Which ancient civilization was known for its advanced engineering and legal systems?"
]

for i, query in enumerate(queries):
    print(f"\n--- Query {i+1}: {query} ---")
    response = router_query_engine.query(query)
    print(f"Response: {response}\n")
    
    # Inspecting source nodes to verify routing
    # The source nodes will typically come from the selected query engine.
    if response.source_nodes:
        # The file_path metadata is added by SimpleDirectoryReader
        source_file = response.source_nodes[0].metadata.get('file_path', 'Unknown Source')
        print(f"Retrieved from source: {Path(source_file).name}")
    else:
        print("No source nodes found (might be a direct LLM answer or routing issue).")

print("\n--- Multi-Document Query Engine Demonstration Complete ---")

# Clean up mock documents
# import shutil
# shutil.rmtree(data_dir)
# print(f"Cleaned up mock documents directory: {data_dir}")
